In [125]:
from config.ingestion.brokers import KiteConfig
import polars as pl
import polars.selectors as cs

In [126]:
stock_table_id = KiteConfig.get_db_tbl(
    data_type=KiteConfig.TYP_STOCKS, frequency="day", failed_tbl=False
)

query = f"""
    select *
    from {stock_table_id}
"""
data = pl.read_database_uri(query=query, uri=KiteConfig.DB_CONN)

In [ ]:
import polars as pl

# Updated Settings
SMA_DAYS = [20, 50]
HL_DAYS = [20, 50]
MICRO_WINDOW = 20


def rolling_percentile(expr: pl.Expr, window: int) -> pl.Expr:
    return expr.rolling_rank(window_size=window) / window

In [128]:
breath_df = (
    data.lazy()
    .with_columns([pl.col("timestamp").cast(pl.Date)])
    .with_columns(
        [
            pl.col("close")
            .rolling_mean(window_size=n)
            .over(partition_by="symbol", order_by="timestamp", descending=False)
            .round(2)
            .alias(f"close_sma_{n}")
            for n in SMA_DAYS
        ]
        + [
            pl.col("close")
            .rolling_max(n)
            .over(partition_by="symbol", order_by="timestamp", descending=False)
            .round(2)
            .alias(f"close_high_{n}")
            for n in HL_DAYS
        ]
        + [
            pl.col("close")
            .rolling_min(n)
            .over(partition_by="symbol", order_by="timestamp", descending=False)
            .round(2)
            .alias(f"close_low_{n}")
            for n in HL_DAYS
        ]
        + [
            pl.col("close")
            .shift(1)
            .over(partition_by="symbol", order_by="timestamp", descending=False)
            .alias("close_prev_1")
        ]
    )
    .with_columns(
        [
            (pl.col("close") > pl.col(f"close_sma_{n}"))
            .cast(pl.Int8)
            .alias(f"above_{n}")
            for n in SMA_DAYS
        ]
        + [
            (pl.col("close") == pl.col(f"close_high_{n}"))
            .cast(pl.Int8)
            .alias(f"new_high_{n}")
            for n in HL_DAYS
        ]
        + [
            (pl.col("close") == pl.col(f"close_low_{n}"))
            .cast(pl.Int8)
            .alias(f"new_low_{n}")
            for n in HL_DAYS
        ]
        + [(pl.col("close") > pl.col("close_prev_1")).cast(pl.Int8).alias("advance")]
        + [(pl.col("close") < pl.col("close_prev_1")).cast(pl.Int8).alias("decline")]
    )
    .group_by("timestamp")
    .agg(
        [
            pl.col(f"above_{n}").mean().round(4).alias(f"pct_above_sma_{n}")
            for n in SMA_DAYS
        ]
        + [
            pl.col(f"new_high_{n}").mean().round(4).alias(f"pct_new_high_{n}")
            for n in HL_DAYS
        ]
        + [
            pl.col(f"new_low_{n}").mean().round(4).alias(f"pct_new_low_{n}")
            for n in HL_DAYS
        ]
        + [pl.col(f"new_high_{n}").sum().alias(f"nh_{n}") for n in HL_DAYS]
        + [pl.col(f"new_low_{n}").sum().alias(f"nl_{n}") for n in HL_DAYS]
        + [pl.col("advance").sum().alias("adv")]
        + [pl.col("decline").sum().alias("dec")]
        + [pl.col("symbol").len().alias("symbol_count")]
    )
    .with_columns(
        [
            (pl.col(f"nh_{n}") / pl.col(f"nl_{n}")).round(4).alias(f"nh_nl_ratio_{n}")
            for n in HL_DAYS
        ]
        + [(pl.col("adv") / pl.col("dec")).round(4).alias("ad_ratio")]
    )
    .sort("timestamp")
    .with_columns(
        [
            # 1. Hyper-Sensitive Percentiles
            rolling_percentile(pl.col("pct_above_sma_20"), MICRO_WINDOW).alias(
                "p_trend"
            ),
            rolling_percentile(pl.col("pct_new_high_20"), MICRO_WINDOW).alias(
                "p_momentum"
            ),
            # Use +1 in denominator to prevent Division-by-Zero
            rolling_percentile(
                pl.col("nh_20") / (pl.col("nl_50") + 1), MICRO_WINDOW
            ).alias("p_expansion"),
            rolling_percentile(pl.col("adv") / (pl.col("dec") + 1), MICRO_WINDOW).alias(
                "p_participation"
            ),
        ]
    )
    .with_columns(
        [
            # 2. Final Breadth Calculation
            (
                (0.30 * pl.col("p_trend"))
                + (0.30 * pl.col("p_momentum"))
                + (0.20 * pl.col("p_expansion"))
                + (0.20 * pl.col("p_participation"))
            ).alias("breadth_raw")
        ]
    )
    .with_columns(
        [
            # 3. Smoothing (Span 3 for even faster reaction)
            (pl.col("breadth_raw").ewm_mean(span=21) * 2 - 1).alias("final_score_20")
        ]
    )
    .with_columns(
        [
            # 3. Triple EMA Smoothing
            pl.col("breadth_raw").ewm_mean(span=5).alias("ema_5"),  # Fast
            pl.col("breadth_raw").ewm_mean(span=10).alias("ema_10"),  # Medium
            pl.col("breadth_raw").ewm_mean(span=20).alias("ema_20"),  # Slow
        ]
    )
    .with_columns(
        [
            # 4. Convert to Symmetric Scale (-1 to +1)
            (2 * pl.col("ema_5") - 1).alias("final_fast"),
            (2 * pl.col("ema_10") - 1).alias("final_medium"),
            (2 * pl.col("ema_20") - 1).alias("final_slow"),
        ]
    )
    .with_columns(
        [
            # 5. Momentum (Slope)
            (pl.col("final_fast") - pl.col("final_medium")).alias("breadth_momentum")
        ]
    )
    .select("timestamp", "final_fast", "final_medium", "final_slow", "breadth_momentum")
    .remove(pl.all_horizontal(cs.exclude("timestamp").is_null()))
    .with_columns(
        [
            pl.when(pl.col("final_medium") >= 0.6)
            .then(pl.lit("🚀 Strong Bull"))
            .when(pl.col("final_medium") >= 0.2)
            .then(pl.lit("📈 Bullish"))
            .when(pl.col("final_medium") > -0.2)
            .then(pl.lit("⚖️ Neutral"))
            .when(pl.col("final_medium") > -0.6)
            .then(pl.lit("📉 Bearish"))
            .otherwise(pl.lit("🔥 Strong Bear"))
            .alias("regime_label")
        ]
    )
    .with_columns(
        [
            pl.when((pl.col("final_medium") > 0.2) & (pl.col("breadth_momentum") > 0))
            .then(pl.lit("🟩 Long (Strengthening)"))
            .when((pl.col("final_medium") > 0.2) & (pl.col("breadth_momentum") < 0))
            .then(pl.lit("🟨 Long (Weakening)"))
            .when((pl.col("final_medium") < -0.2) & (pl.col("breadth_momentum") < 0))
            .then(pl.lit("🟥 Short (Strengthening)"))
            .when((pl.col("final_medium") < -0.2) & (pl.col("breadth_momentum") > 0))
            .then(pl.lit("🟧 Short (Weakening)"))
            .otherwise(pl.lit("⬜ Choppy / No Trade"))
            .alias("trade_recommendation")
        ]
    ).sort("timestamp")
    .select("timestamp", "regime_label", "trade_recommendation")
    .with_columns(pl.col("timestamp").dt.strftime("%d %b %y"))
)

In [129]:
breath_df.collect().write_csv(f"test_{MICRO_WINDOW}.csv")

In [130]:
breath_df.filter(pl.col("regime_label").str.contains("Strong")).collect()

timestamp,regime_label,trade_recommendation
str,str,str
"""19 Jun 25""","""🔥 Strong Bear""","""⬜ Choppy / No Trade"""
"""28 Jul 25""","""🔥 Strong Bear""","""🟥 Short (Strengthening)"""
"""31 Jul 25""","""🔥 Strong Bear""","""🟥 Short (Strengthening)"""
"""01 Aug 25""","""🔥 Strong Bear""","""🟥 Short (Strengthening)"""
"""04 Aug 25""","""🔥 Strong Bear""","""🟥 Short (Strengthening)"""
…,…,…
"""14 Aug 25""","""🔥 Strong Bear""","""🟧 Short (Weakening)"""
"""17 Sep 25""","""🚀 Strong Bull""","""🟩 Long (Strengthening)"""
"""18 Sep 25""","""🚀 Strong Bull""","""🟩 Long (Strengthening)"""
